# Lab 0 - Creating Training Data


## Random Split
### Step 1 - Load Data

In [34]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Foto/Titanic-Dataset-Fixed.csv')

df2 = df.copy()
df3 = df.copy()
df4 = df.copy()

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,DECK,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,DECK,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,DECK,S


### Step 2 - Split Data (8:1:1)

In [16]:
from sklearn.model_selection import train_test_split

df_train, df_unseen = train_test_split(df, test_size=0.2, random_state=0)
df_val, df_test = train_test_split(df_unseen, test_size=0.5, random_state=0)

print(f'Original data count: {df.shape[0]}')
print(f'Training data count: {df_train.shape[0]}')
print(f'Validation data count: {df_val.shape[0]}')
print(f'Test data count: {df_test.shape[0]}')

print('=========')
print(f'Original label counts:\n{df.Survived.value_counts()}')
print(f'Training label counts:\n{df_train.Survived.value_counts()}')
print(f'Validation label counts:\n{df_val.Survived.value_counts()}')
print(f'Test label counts:\n{df_test.Survived.value_counts()}')

Original data count: 891
Training data count: 712
Validation data count: 89
Test data count: 90
Original label counts:
Survived
0    549
1    342
Name: count, dtype: int64
Training label counts:
Survived
0    439
1    273
Name: count, dtype: int64
Validation label counts:
Survived
0    53
1    36
Name: count, dtype: int64
Test label counts:
Survived
0    57
1    33
Name: count, dtype: int64


## Stratified Split


### Step 1 - Load Data

In [36]:

df2 = pd.read_csv('/content/drive/MyDrive/Foto/Titanic-Dataset-Fixed.csv')
df2.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,DECK,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,DECK,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,DECK,S


### Step 2 - Split Data (8:1:1)

In [37]:
df2_train, df2_unseen = train_test_split(
    df2, test_size=0.2, random_state=0, stratify=df2['Survived'])

df2_val, df2_test = train_test_split(
    df2_unseen, test_size=0.5, random_state=0, stratify=df2_unseen['Survived'])

print(f'Original label counts:\n{df2.Survived.value_counts()}')
print(f'Training label counts:\n{df2_train.Survived.value_counts()}')
print(f'Validation label counts:\n{df2_val.Survived.value_counts()}')
print(f'Test label counts:\n{df2_test.Survived.value_counts()}')

Original label counts:
Survived
0    549
1    342
Name: count, dtype: int64
Training label counts:
Survived
0    439
1    273
Name: count, dtype: int64
Validation label counts:
Survived
0    55
1    34
Name: count, dtype: int64
Test label counts:
Survived
0    55
1    35
Name: count, dtype: int64


In [ ]:
def ratio(d): return round(d.Survived.mean(), 3)
pd.DataFrame({
    'random':     [ratio(df_train), ratio(df_val), ratio(df_test)],
    'stratified': [ratio(df2_train), ratio(df2_val), ratio(df2_test)],
}, index=['train', 'val', 'test'])

## Cross Validation 1


In [39]:
df3 = pd.read_csv('/content/drive/MyDrive/Foto/Titanic-Dataset-Fixed.csv')
df3.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,DECK,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,DECK,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,DECK,S


### Step 2 - Split Data

In [40]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=4)
print(f'Number of folds: {kf.get_n_splits()}')
print(f'KFold object: {kf}')
print(f'Number of rows in df3: {df3.shape[0]}')

# Print a compact summary of each fold instead of every index
for i, (train_index, test_index) in enumerate(kf.split(df3), start=1):
    print(f'Fold {i}: train={len(train_index)} rows '
          f'(first/last idx {train_index[0]}/{train_index[-1]}) | '
          f'test={len(test_index)} rows (idx {test_index[0]}-{test_index[-1]})')

Number of folds: 4
KFold object: KFold(n_splits=4, random_state=None, shuffle=False)
Number of rows in df3: 891
Fold 1: train=668 rows (first/last idx 223/890) | test=223 rows (idx 0-222)
Fold 2: train=668 rows (first/last idx 0/890) | test=223 rows (idx 223-445)
Fold 3: train=668 rows (first/last idx 0/890) | test=223 rows (idx 446-668)
Fold 4: train=669 rows (first/last idx 0/668) | test=222 rows (idx 669-890)


In [41]:
kf_shuffled = KFold(n_splits=4, shuffle=True, random_state=0)
for i, (tr, te) in enumerate(kf_shuffled.split(df3), start=1):
    print(f'Fold {i}: train={len(tr)}, test={len(te)}, first test idx={te[:5]}')

Fold 1: train=668, test=223, first test idx=[ 1  5  8 10 14]
Fold 2: train=668, test=223, first test idx=[ 0  2  6 12 15]
Fold 3: train=668, test=223, first test idx=[ 3  4  7 13 16]
Fold 4: train=669, test=222, first test idx=[ 9 11 19 23 24]


## Cross Validation 2


In [43]:
df4 = pd.read_csv('/content/drive/MyDrive/Foto/Titanic-Dataset-Fixed.csv')
df4.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,DECK,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,DECK,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,DECK,S


### Step 2 - Split Data


In [44]:
df4_train, df4_test = train_test_split(df4, test_size=0.2, random_state=0)

kf2 = KFold(n_splits=4)
print(f'Number of folds: {kf2.get_n_splits()}')
print(f'KFold object: {kf2}')
print(f'Number of rows in df4_train: {df4_train.shape[0]}')
print(f'Number of rows in df4_test (held out): {df4_test.shape[0]}')

for i, (train_index, val_index) in enumerate(kf2.split(df4_train), start=1):
    print(f'Fold {i}: train={len(train_index)} | validation={len(val_index)} '
          f'(idx {val_index[0]}-{val_index[-1]})')

Number of folds: 4
KFold object: KFold(n_splits=4, random_state=None, shuffle=False)
Number of rows in df4_train: 712
Number of rows in df4_test (held out): 179
Fold 1: train=534 | validation=178 (idx 0-177)
Fold 2: train=534 | validation=178 (idx 178-355)
Fold 3: train=534 | validation=178 (idx 356-533)
Fold 4: train=534 | validation=178 (idx 534-711)
